In [6]:
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
import pickle
import time
import re

loader=DirectoryLoader("./Data",glob="**/*")
docs=loader.load()

def cleaner_func(docs):
    cleaned = []
    for doc in docs:
        if isinstance(doc, str):
            doc = re.sub(r'\n+', '\n', doc)
            doc = re.sub(r'\s+', ' ', doc)
            cleaned.append(doc.strip())
        else:
            cleaned.append(doc)
    return cleaned

cleaned_docs = cleaner_func(docs)

text_split=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""]
    

)

cleaned_chunks=text_split.split_documents(cleaned_docs)
embd=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store=Chroma.from_documents(
    documents=cleaned_chunks,
    embedding=embd,
    collection_name="Lang_Rag_hf",
    persist_directory="./Rag/chroma_DB"

)

text_chunk=[chunk.page_content for chunk in cleaned_chunks]
tokenized_chunks=[chunk.split() for chunk in text_chunk]

bm25=BM25Okapi(tokenized_chunks)

with open("./Rag/bm25.pkl", 'wb') as f:
    pickle.dump(bm25, f)
with open("./Rag/text_chunks.pkl", 'wb') as f:
    pickle.dump(text_chunk, f)
print(f" BM25 index saved to ./Rag/bm25.pkl")

with open("./Rag/chunks.pkl", 'wb') as f:
    pickle.dump(cleaned_chunks, f)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5429.08it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 BM25 index saved to ./Rag/bm25.pkl
